## stream without structured output

In [13]:
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field 
from typing import List

load_dotenv()

client = OpenAI()


previous_response_id = None
system_prompt = input("Model behaviour or constraints: ")

while True:
    user_input = input("Instructions: ")

    if user_input.lower() in {"exit", "quit"}:
        break

    response = client.responses.create(
        model="gpt-5.4-nano",
        instructions=system_prompt,  
        input=user_input, 
        stream=True,
        previous_response_id=previous_response_id,
        max_output_tokens=512,
    )

    print("Assistant: ", end="", flush=True)

    response_id = None

    for event in response:
        if event.type == "response.output_text.delta":
            print(event.delta, end="", flush=True)
        elif event.type == "response.completed":
            response_id = event.response.id

    print("\n")

    previous_response_id = response_id

Assistant: I don’t have real-time news access, but recent AI momentum includes stronger multimodal models (text+image+audio), faster and cheaper inference via new hardware/optimization, expanding agentic assistants for tool use, and ongoing research on safer alignment, privacy, and reliability (less hallucination).

Assistant: Nice—blue is a great choice. It often symbolizes calm, trust, and stability. What shade of blue is your favorite (navy, sky blue, cobalt, etc.)?

Assistant: Your favourite color is blue.



## stream with structured output 

In [14]:
class ResearchExtraction(BaseModel):
    summary: str = Field(description="A 1-2 sentence concise summary.")
    key_points: List[str] = Field(description="List of core facts or key takeaways.")
    confidence_score: float = Field(description="Confidence rating between 0.0 and 1.0.")

previous_response_id = None
system_prompt = input("Model behaviour or constraints: ")

while True:
    user_input = input("\nInstructions: ")

    if user_input.lower() in {"exit", "quit"}:
        break

    with client.responses.stream(
        model="gpt-4o-mini",
        instructions=system_prompt,
        input=user_input,
        text_format=ResearchExtraction, 
        previous_response_id=previous_response_id,
    ) as stream:

        print("\nAssistant (streaming JSON): ", end="", flush=True)

        for event in stream:
            if event.type == "response.output_text.delta":
                print(event.delta, end="", flush=True)

        print("\n")
        
        final_response = stream.get_final_response()
        
        previous_response_id = final_response.id

        parsed_data: ResearchExtraction = final_response.output_parsed

        print("--- Parsed Output ---")
        print(f"Summary: {parsed_data.summary}")
        print(f"Key Points: {parsed_data.key_points}")
        print(f"Confidence: {parsed_data.confidence_score * 100:.0f}%")


Assistant (streaming JSON): {"summary":"Recent advancements in AI include breakthroughs in natural language processing, ethical frameworks for AI development, and increased funding for AI research.","key_points":["AI models are becoming more capable of understanding and generating human-like text.","Major companies are developing ethical guidelines to address biases in AI systems.","The AI industry is seeing significant investment from both private and public sectors."],"confidence_score":0.9}

--- Parsed Output ---
Summary: Recent advancements in AI include breakthroughs in natural language processing, ethical frameworks for AI development, and increased funding for AI research.
Key Points: ['AI models are becoming more capable of understanding and generating human-like text.', 'Major companies are developing ethical guidelines to address biases in AI systems.', 'The AI industry is seeing significant investment from both private and public sectors.']
Confidence: 90%


## With custom tool web search

In [15]:
import json
from dotenv import load_dotenv
from typing import List
from openai import OpenAI
from pydantic import BaseModel, Field
from ddgs import DDGS

load_dotenv()
client = OpenAI()

class ResearchExtraction(BaseModel):
    summary: str = Field(description="A 1-2 sentence concise summary.")
    key_points: List[str] = Field(description="List of core facts or key takeaways.")
    confidence_score: float = Field(description="Confidence rating between 0.0 and 1.0.")


class CustomSearchArgs(BaseModel):
    query: str = Field(description="The search query to look up on the web.")
    max_results: int = Field(default=2, description="Number of search results to return.")

def custom_web_search(query: str, max_results: int = 2) -> str:
    """Searches the web for up-to-date real-time information on a topic."""
    print(f"\nTool Triggered: Running real DuckDuckGo search for '{query}'...")
    
    try:
        
        results = DDGS().text(query, max_results=max_results)
        
        formatted_results = [
            {
                "title": r.get("title", ""),
                "snippet": r.get("body", "")  
            }
            for r in results
        ]
        
        return json.dumps(formatted_results)

    except Exception as e:
        print(f"Search Error: {e}")
        return json.dumps([{"error": "Failed to fetch live search results."}])

# Required if using raw APIs directly, if there's a high level function that turns python into schema dictionary then we can directly pass custom_web_search
CUSTOM_SEARCH_TOOL = {
    "type": "function",
    "name": "custom_web_search",
    "description": custom_web_search.__doc__,
    "parameters": CustomSearchArgs.model_json_schema()
}

available_tools = {
    "custom_web_search": custom_web_search
}


previous_response_id = None
system_prompt = input("Model behaviour or constraints: ")

while True:
    user_input = input("\nInstructions: ")

    if user_input.lower() in {"exit", "quit"}:
        break

    with client.responses.stream(
        model="gpt-4o-mini",
        instructions=system_prompt,
        input=user_input,
        text_format=ResearchExtraction,  
        tools=[CUSTOM_SEARCH_TOOL],      
        previous_response_id=previous_response_id,
    ) as stream:

        print("\nAssistant (streaming): ", end="", flush=True)

        for event in stream:
            if event.type == "response.output_text.delta":
                print(event.delta, end="", flush=True)

        final_response = stream.get_final_response()
        previous_response_id = final_response.id


    tool_calls = [
        item for item in (final_response.output or []) 
        if getattr(item, "type", None) == "function_call"
    ]

    if tool_calls:
        for tool_call in tool_calls:
            fn_name = tool_call.name
            fn_args = json.loads(tool_call.arguments)
            
            if fn_name in available_tools:
                tool_output_str = available_tools[fn_name](**fn_args)
                
                tool_output_item = {
                    "type": "function_call_output",
                    "call_id": tool_call.call_id,
                    "output": tool_output_str
                }
                
                with client.responses.stream(
                    model="gpt-4o-mini",
                    instructions=system_prompt,
                    input=[tool_output_item], 
                    text_format=ResearchExtraction,
                    previous_response_id=previous_response_id,
                ) as tool_stream:
                    
                    print("\nAssistant (synthesizing tool results): ", end="", flush=True)
                    for event in tool_stream:
                        if event.type == "response.output_text.delta":
                            print(event.delta, end="", flush=True)
                    
                    final_response = tool_stream.get_final_response()
                    previous_response_id = final_response.id

    parsed_data: ResearchExtraction = final_response.output_parsed

    if parsed_data:
        print("\n\n--- Final Parsed Output ---")
        print(f"Summary: {parsed_data.summary}")
        print(f"Key Points: {parsed_data.key_points}")
        print(f"Confidence: {parsed_data.confidence_score * 100:.0f}%\n")


Assistant (streaming): 
Tool Triggered: Running real DuckDuckGo search for 'latest news on AI'...

Assistant (synthesizing tool results): {"summary":"Recent developments in AI highlight advancements in various sectors, with a focus on ethical implications and business applications.","key_points":["A surge in AI-driven business growth reported, highlighting new technologies and innovations.","TechCrunch emphasizes the importance of discussing ethical issues surrounding AI advancements.","WIRED covers the latest AI technologies, offering insights into their impact on society.","OpenAI announces new initiatives including programs tailored for small businesses.","Topics of recent AI news include machine learning, deep learning, and their applications in numerous industries."],"confidence_score":0.9}

--- Final Parsed Output ---
Summary: Recent developments in AI highlight advancements in various sectors, with a focus on ethical implications and business applications.
Key Points: ['A surge

## Debugged version to check the details

In [16]:
import json
from dotenv import load_dotenv
from typing import List
from openai import OpenAI
from pydantic import BaseModel, Field
from ddgs import DDGS

load_dotenv()
client = OpenAI()

class ResearchExtraction(BaseModel):
    summary: str = Field(description="A 1-2 sentence concise summary.")
    key_points: List[str] = Field(description="List of core facts or key takeaways.")
    confidence_score: float = Field(description="Confidence rating between 0.0 and 1.0.")

class CustomSearchArgs(BaseModel):
    query: str = Field(description="The search query to look up on the web.")
    max_results: int = Field(default=2, description="Number of search results to return.")

def custom_web_search(query: str, max_results: int = 2) -> str:
    """Searches the web for up-to-date real-time information on a topic."""
    print(f"\nTool Triggered: Running real DuckDuckGo search for '{query}'...")
    
    try:
        results = DDGS().text(query, max_results=max_results)
        
        formatted_results = [
            {
                "title": r.get("title", ""),
                "snippet": r.get("body", "")  
            }
            for r in results
        ]
        
        return json.dumps(formatted_results)

    except Exception as e:
        print(f"Search Error: {e}")
        return json.dumps([{"error": "Failed to fetch live search results."}])

CUSTOM_SEARCH_TOOL = {
    "type": "function",
    "name": "custom_web_search",
    "description": custom_web_search.__doc__,
    "parameters": CustomSearchArgs.model_json_schema()
}

available_tools = {
    "custom_web_search": custom_web_search
}

previous_response_id = None
system_prompt = input("Model behaviour or constraints: ")

while True:
    user_input = input("\nInstructions: ")

    if user_input.lower() in {"exit", "quit"}:
        break

    with client.responses.stream(
        model="gpt-4o-mini",
        instructions=system_prompt,
        input=user_input,
        text_format=ResearchExtraction,  
        tools=[CUSTOM_SEARCH_TOOL],      
        previous_response_id=previous_response_id,
    ) as stream:

        print("\nAssistant (streaming): ", end="", flush=True)

        for event in stream:
            if event.type == "response.output_text.delta":
                print(event.delta, end="", flush=True)

        final_response = stream.get_final_response()
        previous_response_id = final_response.id

    # 🔍 DEBUG PRINT: Show initial final_response object
    print(f"\n\n[DEBUG] Initial final_response Object:\n{final_response}")

    # Extract tool calls from response
    tool_calls = [
        item for item in (final_response.output or []) 
        if getattr(item, "type", None) == "function_call"
    ]

    print(f"\n[DEBUG] Extracted Tool Calls: {tool_calls}")

    if tool_calls:
        for tool_call in tool_calls:
            fn_name = tool_call.name
            fn_args = json.loads(tool_call.arguments)
            
            print(f"\n[DEBUG] Tool requested: {fn_name}")
            print(f"[DEBUG] Parsed Arguments (fn_args): {fn_args}")

            if fn_name in available_tools:
                tool_output_str = available_tools[fn_name](**fn_args)
                print(f"[DEBUG] Tool Raw Output (tool_output_str):\n{tool_output_str}")

                tool_output_item = {
                    "type": "function_call_output",
                    "call_id": tool_call.call_id,
                    "output": tool_output_str
                }

                print(f"[DEBUG] Tool Output Item (tool_output_item):\n{json.dumps(tool_output_item, indent=2)}")

                with client.responses.stream(
                    model="gpt-4o-mini",
                    instructions=system_prompt,
                    input=[tool_output_item], 
                    text_format=ResearchExtraction,
                    previous_response_id=previous_response_id,
                ) as tool_stream:
                    
                    print("\nAssistant (synthesizing tool results): ", end="", flush=True)
                    for event in tool_stream:
                        if event.type == "response.output_text.delta":
                            print(event.delta, end="", flush=True)
                    
                    final_response = tool_stream.get_final_response()
                    previous_response_id = final_response.id

                    # 🔍 DEBUG PRINT: Show updated final_response object after synthesis
                    print(f"\n\n[DEBUG] Post-Tool Synthesis final_response Object:\n{final_response}")

    parsed_data: ResearchExtraction = final_response.output_parsed

    print(f"\n[DEBUG] Pydantic Parsed Data Object: {parsed_data}")

    if parsed_data:
        print("\n\n--- Final Parsed Output ---")
        print(f"Summary: {parsed_data.summary}")
        print(f"Key Points: {parsed_data.key_points}")
        print(f"Confidence: {parsed_data.confidence_score * 100:.0f}%\n")


Assistant (streaming): 

[DEBUG] Initial final_response Object:
ParsedResponse[~TextFormatT](id='resp_00ea2803d820f4d4006a664b50da70819caef38d5e2aa5d4b5', created_at=1785088848.0, error=None, incomplete_details=None, instructions='act like a research assistant', metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ParsedResponseFunctionToolCall(arguments='{"query":"latest news on AI","max_results":5}', call_id='call_qxwBUTP06tMHYrC1yPJ1XLyg', name='custom_web_search', type='function_call', id='fc_00ea2803d820f4d4006a664b51680c819ca1c0e7b9f11161e3', caller=None, namespace=None, status='completed', parsed_arguments=None)], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='custom_web_search', parameters={'properties': {'query': {'description': 'The search query to look up on the web.', 'title': 'Query', 'type': 'string'}, 'max_results': {'default': 2, 'description': 'Number of search results to return.', 'title': 'Max Results', 't

## agentic workflow added (simple reflection step that checks the summary length)

In [3]:
import json
from dotenv import load_dotenv
from typing import List
from openai import OpenAI
from pydantic import BaseModel, Field
from ddgs import DDGS

load_dotenv()
client = OpenAI()

class ResearchExtraction(BaseModel):
    summary: str = Field(description="A 1-2 sentence concise summary.")
    key_points: List[str] = Field(description="List of core facts or key takeaways.")
    confidence_score: float = Field(description="Confidence rating between 0.0 and 1.0.")

class CustomSearchArgs(BaseModel):
    query: str = Field(description="The search query to look up on the web.")
    max_results: int = Field(default=2, description="Number of search results to return.")

def is_length_valid(summary: str, max_chars: int = 90) -> tuple[bool, str]:
    """Checks summary length and returns a status + feedback message."""
    if len(summary) > max_chars:
        feedback = f"Your summary was too long ({len(summary)} characters). It MUST be under {max_chars} characters."
        return False, feedback
    return True, f"The summary is under {max_chars} characters which is fine"

def custom_web_search(query: str, max_results: int = 2) -> str:
    """Searches the web for up-to-date real-time information on a topic."""
    print(f"\nTool Triggered: Running real DuckDuckGo search for '{query}'...")
    try:
        results = DDGS().text(query, max_results=max_results)
        formatted_results = [
            {
                "title": r.get("title", ""),
                "snippet": r.get("body", "")  
            }
            for r in results
        ]
        return json.dumps(formatted_results)
    except Exception as e:
        print(f"Search Error: {e}")
        return json.dumps([{"error": "Failed to fetch live search results."}])

CUSTOM_SEARCH_TOOL = {
    "type": "function",
    "name": "custom_web_search",
    "description": custom_web_search.__doc__,
    "parameters": CustomSearchArgs.model_json_schema()
}

available_tools = {
    "custom_web_search": custom_web_search
}

previous_response_id = None
system_prompt = input("Model behaviour or constraints: ")

while True:
    user_input = input("\nInstructions: ")

    if user_input.lower() in {"exit", "quit"}:
        break

    current_prompt = user_input
    max_retries = 3

    for attempt in range(max_retries):
        
        with client.responses.stream(
            model="gpt-4o-mini",
            instructions=system_prompt,
            input=current_prompt,
            text_format=ResearchExtraction,  
            tools=[CUSTOM_SEARCH_TOOL],      
            previous_response_id=previous_response_id,
        ) as stream:

            print(f"\nAssistant (streaming - attempt {attempt + 1}): ", end="", flush=True)

            for event in stream:
                if event.type == "response.output_text.delta":
                    print(event.delta, end="", flush=True)

            final_response = stream.get_final_response()
            previous_response_id = final_response.id

        tool_calls = [
            item for item in (final_response.output or []) 
            if getattr(item, "type", None) == "function_call"
        ]

        if tool_calls:
            for tool_call in tool_calls:
                fn_name = tool_call.name
                fn_args = json.loads(tool_call.arguments)
                
                if fn_name in available_tools:
                    tool_output_str = available_tools[fn_name](**fn_args)
                    
                    tool_output_item = {
                        "type": "function_call_output",
                        "call_id": tool_call.call_id,
                        "output": tool_output_str
                    }
                    
                    with client.responses.stream(
                        model="gpt-4o-mini",
                        instructions=system_prompt,
                        input=[tool_output_item], 
                        text_format=ResearchExtraction,
                        previous_response_id=previous_response_id,
                    ) as tool_stream:
                        
                        print("\nAssistant (synthesizing tool results): ", end="", flush=True)
                        for event in tool_stream:
                            if event.type == "response.output_text.delta":
                                print(event.delta, end="", flush=True)
                        
                        final_response = tool_stream.get_final_response()
                        previous_response_id = final_response.id

        parsed_data: ResearchExtraction = final_response.output_parsed

        if parsed_data:
            valid, feedback = is_length_valid(parsed_data.summary, max_chars=90)
            
            if valid:
                print("\n\n--- Final Validated Output ---")
                print(f"Summary ({len(parsed_data.summary)} chars): {parsed_data.summary}")
                print(f"Key Points: {parsed_data.key_points}")
                print(f"Confidence: {parsed_data.confidence_score * 100:.0f}%\n")
                break
            else:
                print(f"\n\nValidation Failed: {feedback}")
                print("Re-running turn automatically with feedback...")
                current_prompt = f"RETRY: {feedback}. Please rewrite the summary to be shorter."
        else:
            break


Assistant (streaming - attempt 1): 
Tool Triggered: Running real DuckDuckGo search for 'latest AI news'...

Assistant (synthesizing tool results): {"summary":"The latest news on artificial intelligence includes updates on breakthroughs, ethical issues, and developments in AI technologies and applications across various industries.","key_points":["Google News provides a broad overview of artificial intelligence articles and updates.","AI News focuses on insights driving business growth through AI technologies.","TechCrunch covers trends, companies, and ethical challenges in AI and machine learning.","Reuters offers updates on AI breakthroughs, regulations, and their global impact.","AI Weekly curates news, funding updates, and model releases for AI professionals."],"confidence_score":0.9}

Validation Failed: Your summary was too long (173 characters). It MUST be under 90 characters.
Re-running turn automatically with feedback...

Assistant (streaming - attempt 2): {"summary":"Latest AI

## With openai agent sdk

In [9]:
import json, asyncio
from typing import List
from dotenv import load_dotenv
from ddgs import DDGS
from pydantic import BaseModel
from agents import Agent, Runner, function_tool

load_dotenv()

class ResearchExtraction(BaseModel):
    summary: str
    key_points: List[str]
    confidence_score: float

class ReviewResult(BaseModel):
    approved: bool
    feedback: str

@function_tool
def custom_web_search(query: str, max_results: int = 2) -> str:
    """Search the web using DuckDuckGo."""
    results = DDGS().text(query, max_results=max_results)
    return json.dumps([{"title": r.get("title", ""), "snippet": r.get("body", "")} for r in results])

research_agent = Agent(
    name="Research Agent",
    instructions="Professional research assistant. Use search for recent info. Return concise summary, key points, and confidence.",
    tools=[custom_web_search],
    output_type=ResearchExtraction,
)

review_agent = Agent(
    name="Reviewer",
    instructions="Review research. Approve (true) if summary < 90 chars, points complete, and confidence reasonable. Otherwise approve=false with feedback.",
    output_type=ReviewResult,
)

async def research_workflow(user_query: str, max_revisions: int = 3):
    prompt = user_query
    last_response_id = None

    for i in range(max_revisions):
        print(f"\n--- Iteration {i+1} ---")
        
        research = await Runner.run(research_agent, prompt, previous_response_id=last_response_id)
        last_response_id = research.last_response_id
        res_output = research.final_output

        review = await Runner.run(
            review_agent, 
            f"User request: {user_query}\nResearch output:\n{res_output.model_dump_json(indent=2)}"
        )
        rev_output: ReviewResult = review.final_output
        print(f"Reviewer: {rev_output.feedback}")

        if rev_output.approved:
            print("Approved")
            return res_output

        prompt = f"The reviewer gave feedback:\n{rev_output.feedback}\nRewrite your response for: {user_query}"

    return res_output

async def main():
    while (query := input("\nQuestion: ")) != "quit":
        answer = await research_workflow(query)
        print("\n========== FINAL ==========\n", answer.model_dump_json(indent=2))

if __name__ == "__main__":
    try:
        asyncio.run(main())
    except RuntimeError:
        await main()


--- Iteration 1 ---
Reviewer: Summary exceeds 90 chars. Points are generally complete, confidence is reasonable.

--- Iteration 2 ---
Reviewer: Looks good.
Approved

========== FINAL ==========
 {
  "summary": "AI news: model updates, big compute deals, IPO rumors, and EU rule changes.",
  "key_points": [
    "OpenAI is reportedly preparing a confidential IPO filing.",
    "Anthropic is reported to have a major compute capacity deal.",
    "Google and Meta keep rolling out new AI features and model updates.",
    "EU AI rules are still being adjusted and delayed in parts.",
    "AI security and labor impacts remain active concerns."
  ],
  "confidence_score": 0.68
}


C:\Users\YeXiaoJun\AppData\Local\Temp\ipykernel_21688\1088899854.py:73: RuntimeWarning: coroutine 'main' was never awaited
  await main()
